[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S10/2_Practica/en_clase/S10_01_red_desde_cero_en_clase.ipynb)

# S10 · 01 — Una red neuronal desde cero, solo con NumPy · versión en clase

⏱ 40 min en clase · Sprint 10 · Redes neuronales bien entrenadas (UD 4.1)

**Contexto TurisData.** TurisData Canarias quiere saber si una red neuronal "vale la pena" frente a los modelos clásicos que ya conoces. Antes de usar Keras, vamos a abrir la caja: construirás una red diminuta con tus propias manos para entender **qué hace** por dentro (así, cuando algo falle en Keras, sabrás por dónde mirar).

**Este cuaderno tiene 3 ejercicios** (1 a 3), los que se hacen en el aula. La red completa, la comprobación de gradientes, el entrenamiento con distintos tamaños de capa, el mini-reto de la tasa de aprendizaje y las preguntas de reflexión están en el cuaderno completo. Sus ejercicios 1, 2 y 3 son los 3 de este.

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

SEMILLA = 42
np.random.seed(SEMILLA)
plt.rcParams["figure.figsize"] = (6, 4)
print("Listo ✔")

### 👀 A2 · EJEMPLO — Una neurona de juguete

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Cuánto vale `z` y cuánto la salida de la neurona?
2. ¿Qué significa esa salida en el contexto de TurisData?
3. ¿Qué le pasa a la salida si `z` es muy grande (por ejemplo 20) o muy negativo (−20)?

In [ ]:
def sigmoide(z):
    """Convierte cualquier número en un valor entre 0 y 1 (sirve como probabilidad)."""
    return 1 / (1 + np.exp(-z))

x = np.array([1.5, 1.0])        # antelación 150 días, tarifa flexible
pesos = np.array([0.8, 1.2])    # inventados: la tarifa flexible pesa más
sesgo = -1.0

z = np.dot(x, pesos) + sesgo    # suma ponderada
print("z =", z, "→ salida de la neurona =", round(float(sigmoide(z)), 3))

### ✏️ A3 · EJERCICIO 1 de 3 — La función neurona

**Tu turno:** Escribe la función `neurona(x, pesos, sesgo)` que devuelva la salida de la neurona (suma ponderada + sesgo, pasada por `sigmoide`).

**Debe salir:** ✔ Ejercicio 1 correcto (`neurona(...)` da ≈ 0.802 con el ejemplo y 0.5 con entradas 0)

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A3).

1. ¿Qué devuelve `neurona` con las entradas del ejemplo?
2. ¿Qué daría con un sesgo de −3 en vez de −1?
3. ¿Por qué con entradas 0 y sesgo 0 sale exactamente 0,5?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A3, **después de intentarlo**.

In [ ]:
# pista: usa np.dot(x, pesos) + sesgo y aplica sigmoide al resultado
def neurona(x, pesos, sesgo):
    ...

salida_ejemplo = neurona(np.array([1.5, 1.0]), np.array([0.8, 1.2]), -1.0)
assert abs(salida_ejemplo - 0.8022) < 1e-3, "Revisa neurona(): debe ser sigmoide(x·pesos + sesgo). Esperado ≈ 0.802"
assert neurona(np.array([0.0, 0.0]), np.array([5.0, 5.0]), 0.0) == 0.5, "Con entradas 0 y sesgo 0 la sigmoide da 0.5"
print("✔ Ejercicio 1 correcto")

### 👀 A4 · EJEMPLO — Tres funciones de activación

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A4).

1. ¿Entre qué valores salen la sigmoide, la tanh y la ReLU?
2. ¿Qué activación se usa en la capa final de una clasificación binaria y por qué?
3. ¿Qué ocurre si apilamos capas sin activación?

In [ ]:
def tanh(z):
    return np.tanh(z)

z = np.linspace(-6, 6, 200)
plt.plot(z, sigmoide(z), label="sigmoide")
plt.plot(z, tanh(z), label="tanh")
plt.plot(z, np.maximum(0, z), label="ReLU")
plt.ylim(-1.5, 3); plt.axhline(0, color="gray", lw=0.5); plt.axvline(0, color="gray", lw=0.5)
plt.xlabel("valor de entrada z"); plt.ylabel("salida de la activación")
plt.title("Tres funciones de activación"); plt.legend(); plt.show()

### ✏️ A5 · EJERCICIO 2 de 3 — ReLU y su derivada

**Tu turno:** Define `relu(z)` (devuelve `z` si es positivo y 0 si no) y `relu_derivada(z)` (vale 1 donde `z > 0` y 0 en el resto). La derivada nos dirá más adelante "por dónde fluye el aprendizaje": una neurona ReLU apagada (z ≤ 0) no aprende en esa muestra.

**Debe salir:** ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A5).

1. ¿Qué devuelven `relu` y `relu_derivada` para `z = -2`, `0` y `3`?
2. ¿Por qué `relu_derivada(0)` es 0 y no 1?
3. ¿Qué significa que una neurona ReLU esté apagada (`z ≤ 0`) en una muestra?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A5, **después de intentarlo**.

In [ ]:
# pista: np.maximum(0, z) para relu; (z > 0) convertido a float para la derivada
def relu(z):
    ...

def relu_derivada(z):
    ...

_prueba = np.array([-2.0, 0.0, 3.0])
assert np.array_equal(relu(_prueba), [0.0, 0.0, 3.0]), "relu debe anular los negativos y dejar pasar los positivos"
assert np.array_equal(relu_derivada(_prueba), [0.0, 0.0, 1.0]), "relu_derivada vale 1 solo si z > 0"
print("✔ Ejercicio 2 correcto")

### 👀 A6 · EJEMPLO — Dos anillos y un modelo lineal

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A6).

1. ¿Qué exactitud tiene el modelo lineal y qué significa?
2. ¿Cuántos puntos se usan para entrenar y cuántos para validar?
3. ¿Por qué una recta no puede separar los dos grupos?

In [ ]:
X, y = make_circles(n_samples=600, noise=0.10, factor=0.45, random_state=SEMILLA)
X_ent, X_val, y_ent, y_val = train_test_split(X, y, test_size=0.3, random_state=SEMILLA, stratify=y)

lineal = LogisticRegression().fit(X_ent, y_ent)
print("Exactitud de un modelo lineal en validación:", round(lineal.score(X_val, y_val), 3))

plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=12)
plt.title("Dos grupos que ninguna recta separa"); plt.xlabel("variable 1"); plt.ylabel("variable 2"); plt.show()

### 👀 A7 · EJEMPLO — La pérdida: entropía cruzada

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A7).

1. ¿Qué pérdida tiene una red que siempre dice 0,5?
2. ¿Por qué fallar con seguridad cuesta 4,605 y acertar con seguridad solo 0,01?
3. ¿Qué quiere decir que la pérdida baje durante el entrenamiento?

In [ ]:
def entropia_cruzada(y_real, prob, eps=1e-9):
    """Pérdida media de una clasificación binaria (menor = mejor)."""
    prob = np.clip(prob, eps, 1 - eps)
    return -np.mean(y_real * np.log(prob) + (1 - y_real) * np.log(1 - prob))

print("Si predices siempre 0.5, la pérdida es", round(entropia_cruzada(np.array([0, 1, 1, 0]), np.full(4, 0.5)), 3), "(= ln 2)")
print("Si aciertas con seguridad (0.99 / 0.01):", round(entropia_cruzada(np.array([1, 0]), np.array([0.99, 0.01])), 3))
print("Si fallas con seguridad (0.01 / 0.99):", round(entropia_cruzada(np.array([1, 0]), np.array([0.01, 0.99])), 3))

### ✏️ A8 · EJERCICIO 3 de 3 — Un paso de descenso de gradiente

**Tu turno:** Escribe `paso_gradiente(parametros, gradientes, tasa)`: recibe dos diccionarios con las mismas claves (`"W1"`, `"b1"`, `"W2"`, `"b2"`) y devuelve un diccionario **nuevo** con cada parámetro actualizado: `nuevo = viejo - tasa * gradiente`. `tasa` es la **tasa de aprendizaje** (el tamaño del pasito).

**Debe salir:** ✔ Ejercicio 3 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A8).

1. ¿Qué valores salen para `W1` y `b1` en la prueba?
2. ¿Por qué se **resta** el gradiente y no se suma?
3. ¿Qué pasaría con una tasa de aprendizaje enorme?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A8, **después de intentarlo**.

In [ ]:
# pista: recorre las claves y resta tasa * gradiente a cada parámetro
def paso_gradiente(parametros, gradientes, tasa):
    ...

_p = {"W1": np.array([1.0, 2.0]), "b1": np.array([0.5])}
_g = {"W1": np.array([0.5, -1.0]), "b1": np.array([1.0])}
_nuevo = paso_gradiente(_p, _g, 0.1)
assert np.allclose(_nuevo["W1"], [0.95, 2.1]) and np.allclose(_nuevo["b1"], [0.4]), "Cada parámetro debe restar tasa * gradiente"
assert _p["W1"][0] == 1.0, "No modifiques el diccionario original: devuelve uno nuevo"
print("✔ Ejercicio 3 correcto")

---

## 🏁 Fin del cuaderno de clase

El **[cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S10/2_Practica/para_casa/S10_01_red_desde_cero.ipynb)** trae la red completa, la comprobación de gradientes, el entrenamiento con distintos tamaños de capa, el mini-reto de la tasa de aprendizaje y las preguntas de reflexión, cada ejercicio con su solución plegada en la Parte B.

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas

- **Qué hace:**
    - Carga `numpy` (la base de toda la red), `matplotlib` para los gráficos y, de `scikit-learn`, `make_circles` (datos de juguete), `LogisticRegression` (el modelo lineal con el que compararemos) y `train_test_split`.
    - `dato(...)` es la función de descarga de datos del curso (aquí no se usa: los datos se generan).
    - Fija la semilla `SEMILLA = 42` y el tamaño por defecto de los gráficos.
- **Qué debe salir:**
    - `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'np' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 1. La neurona

#### A2 · Ejemplo — Una neurona de juguete

**Contexto TurisData.** TurisData Canarias quiere saber si una red neuronal «vale la pena» frente a los modelos clásicos. Antes de usar Keras, abrimos la caja: construirás una red diminuta con tus propias manos para entender **qué hace** por dentro.

**Idea.** Una neurona artificial hace tres cosas:
1. Multiplica cada entrada por un **peso** (cuánto le importa esa entrada).
2. Suma todo y añade un **sesgo** (*bias*, un empujón inicial).
3. Pasa el resultado por una **función de activación** (un "interruptor suave" que decide cuánta señal deja pasar).

Ejemplo de juguete: una neurona que estima la "probabilidad de cancelar" a partir de
la antelación (en cientos de días) y de si la tarifa es flexible (1) o no (0).

- **Qué hace:**
    - `sigmoide(z)` convierte cualquier número en un valor entre 0 y 1 (sirve como probabilidad).
    - `x` son las entradas (antelación de 150 días y tarifa flexible), `pesos` cuánto pesa cada una y `sesgo` el empujón inicial (los valores están inventados).
    - `z = np.dot(x, pesos) + sesgo` es la suma ponderada: 1,5 · 0,8 + 1,0 · 1,2 − 1,0 = 1,4. El `print` muestra `z` y la salida de la neurona, `sigmoide(z)`.
- **Qué debe salir:**
    - `z = 1.4000000000000004 → salida de la neurona = 0.802`.
- **Cómo leerlo:**
    - La neurona «cree» que la probabilidad de cancelar es del 80 %. El `1.4000000000000004` es un detalle de cómo el ordenador guarda los decimales: léelo como 1,4.
- **Si falla:**
    - `NameError: name 'np' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto vale `z` y cuánto la salida de la neurona?* → `z = 1,4` y la salida **0,802**.
2. *¿Qué significa esa salida en el contexto de TurisData?* → Que la neurona estima un **80 %** de probabilidad de cancelar para esa reserva.
3. *¿Qué le pasa a la salida si `z` es muy grande (por ejemplo 20) o muy negativo (−20)?* → La sigmoide **se satura**: se acerca a 1 (para 20) o a 0 (para −20) sin llegar nunca a ellos.

</details>

#### A3 · Ejercicio 1 — La función neurona

- **Qué hace:**
    - `np.dot(x, pesos) + sesgo` calcula la suma ponderada de las entradas más el sesgo (`z`).
    - `sigmoide(...)` la convierte en un número entre 0 y 1. La función devuelve ese valor con `return`.
    - La comprobación prueba la neurona con el ejemplo del texto (debe dar ≈ 0,802) y con entradas 0 y sesgo 0 (debe dar exactamente 0,5).
- **Qué debe salir:**
    - `✔ Ejercicio 1 correcto`.
- **Si falla:**
    - `Revisa neurona(): debe ser sigmoide(x·pesos + sesgo). Esperado ≈ 0.802` → falta la `sigmoide`, o el sesgo, o has multiplicado en vez de usar `np.dot`.
    - `Con entradas 0 y sesgo 0 la sigmoide da 0.5` → la función no usa `sigmoide` sobre el resultado.
    - `NameError: name 'sigmoide' is not defined` → falta ejecutar el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué devuelve `neurona` con las entradas del ejemplo?* → **0,802** (≈ 80 %).
2. *¿Qué daría con un sesgo de −3 en vez de −1?* → `z = 0,8·1,5 + 1,2·1,0 − 3 = −0,6` → `sigmoide(−0,6)` ≈ **0,354**: un sesgo más negativo baja la probabilidad.
3. *¿Por qué con entradas 0 y sesgo 0 sale exactamente 0,5?* → Porque `z = 0` y `sigmoide(0) = 1 / (1 + 1) = 0,5`: sin información, la neurona está indecisa.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
def neurona(x, pesos, sesgo):
    return sigmoide(np.dot(x, pesos) + sesgo)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 2. Funciones de activación

#### A4 · Ejemplo — Tres funciones de activación

**Idea.** Sin activación, apilar capas no sirve de nada: una suma de sumas ponderadas sigue
siendo una suma ponderada (un modelo lineal). La activación introduce **curvas**, y con curvas
la red puede dibujar fronteras complicadas.

- **Sigmoide:** salida entre 0 y 1. Típica en la capa final de una clasificación binaria.
- **ReLU:** deja pasar lo positivo y anula lo negativo. Es la habitual en capas ocultas.
- **Tanh:** como la sigmoide pero entre −1 y 1.

- **Qué hace:**
    - `tanh(z)` es la tangente hiperbólica (de NumPy). Se calculan las tres activaciones sobre 200 valores de `z` entre −6 y 6.
    - `np.maximum(0, z)` es la ReLU: deja pasar lo positivo y anula lo negativo.
    - El resto dibuja las tres curvas con leyenda y ejes (el eje vertical se corta entre −1,5 y 3).
- **Qué debe salir:**
    - Un gráfico con tres curvas: la sigmoide (de 0 a 1, forma de S), la tanh (de −1 a 1, también en S) y la ReLU (0 a la izquierda del cero y una recta creciente a la derecha).
- **Cómo leerlo:**
    - Cada curva dice cuánta señal deja pasar la neurona según su `z`. Las dos «S» se aplanan en los extremos (se saturan); la ReLU no se satura para valores positivos.
- **Si falla:**
    - `NameError: name 'sigmoide' is not defined` → falta ejecutar el ejemplo de la neurona.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Entre qué valores salen la sigmoide, la tanh y la ReLU?* → Sigmoide: **0 a 1**. Tanh: **−1 a 1**. ReLU: **0 en adelante** (sin tope).
2. *¿Qué activación se usa en la capa final de una clasificación binaria y por qué?* → La **sigmoide**: su salida entre 0 y 1 se interpreta como probabilidad.
3. *¿Qué ocurre si apilamos capas sin activación?* → Que **no se gana nada**: una suma de sumas ponderadas sigue siendo una suma ponderada, es decir, un modelo lineal.

</details>

#### A5 · Ejercicio 2 — ReLU y su derivada

- **Qué hace:**
    - `np.maximum(0, z)` compara cada valor con 0 y se queda con el mayor: anula los negativos y deja pasar los positivos (`relu`).
    - `(z > 0).astype(float)` da verdadero/falso para cada valor y lo convierte en 1,0 / 0,0: la derivada de la ReLU (`relu_derivada`), que vale 1 donde la neurona está activa y 0 donde está apagada.
    - La comprobación usa la prueba `[-2, 0, 3]`: `relu` debe dar `[0, 0, 3]` y `relu_derivada`, `[0, 0, 1]`.
- **Qué debe salir:**
    - `✔ Ejercicio 2 correcto`.
- **Si falla:**
    - `relu debe anular los negativos y dejar pasar los positivos` → `relu` no es `np.maximum(0, z)` (¿has devuelto `z` o `max(0, z)` de Python, que no funciona con arrays?).
    - `relu_derivada vale 1 solo si z > 0` → para `z = 0` debe dar 0 (la condición es estrictamente `z > 0`) y debe devolver un array de números, no de booleanos.
    - `NameError: name 'np' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué devuelven `relu` y `relu_derivada` para `z = -2`, `0` y `3`?* → `relu`: **0, 0 y 3**. `relu_derivada`: **0, 0 y 1**.
2. *¿Por qué `relu_derivada(0)` es 0 y no 1?* → Por convención: la derivada vale 1 solo donde `z > 0`; en `z = 0` la neurona no deja pasar señal.
3. *¿Qué significa que una neurona ReLU esté apagada (`z ≤ 0`) en una muestra?* → Que su derivada es 0: **no aprende en esa muestra**, porque el error no puede viajar hacia atrás a través de ella.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
def relu(z):
    return np.maximum(0, z)

def relu_derivada(z):
    return (z > 0).astype(float)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 3. Un problema que un modelo lineal no resuelve

#### A6 · Ejemplo — Dos anillos y un modelo lineal

Generamos dos "anillos" de puntos (uno dentro del otro). Piensa en ellos como clientes de
dos tipos con un comportamiento que no se separa con una línea recta. Una regresión logística
(modelo lineal) se quedará en el azar.

- **Qué hace:**
    - `make_circles(...)` genera 600 puntos en dos «anillos» (uno dentro del otro) con algo de ruido: `X` son las coordenadas e `y` el grupo (0 o 1).
    - `train_test_split(...)` reparte 420 puntos para entrenar (`X_ent`, `y_ent`) y 180 para validar (`X_val`, `y_val`).
    - `LogisticRegression().fit(...)` entrena el modelo lineal y `.score(X_val, y_val)` mide su exactitud en validación.
    - El gráfico dibuja los puntos coloreados por grupo.
- **Qué debe salir:**
    - `Exactitud de un modelo lineal en validación: 0.494` y un gráfico con un anillo de puntos rojos rodeando un círculo de puntos azules.
- **Cómo leerlo:**
    - Una exactitud de 0,494 es **azar** (dos grupos casi iguales): ninguna recta puede separar un anillo del círculo interior.
- **Si falla:**
    - `NameError: name 'train_test_split' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué exactitud tiene el modelo lineal y qué significa?* → **0,494**: acierta como el azar (≈ 50 %).
2. *¿Cuántos puntos se usan para entrenar y cuántos para validar?* → **420** para entrenar y **180** para validar (el 70 % y el 30 % de 600).
3. *¿Por qué una recta no puede separar los dos grupos?* → Porque un grupo **rodea** al otro: cualquier recta deja puntos de los dos grupos a cada lado.

</details>

## 4. Propagación hacia delante y hacia atrás

#### A7 · Ejemplo — La pérdida: entropía cruzada

Nuestra red tendrá: **2 entradas → capa oculta (ReLU) → 1 salida (sigmoide)**.
Una capa es simplemente **muchas neuronas a la vez**, y se calcula con una multiplicación de matrices:

`Z1 = X · W1 + b1` → `A1 = relu(Z1)` → `Z2 = A1 · W2 + b2` → `A2 = sigmoide(Z2)`

`A2` es la probabilidad predicha de pertenecer al grupo 1.

**Propagación hacia atrás.** Para aprender medimos el error (pérdida) y preguntamos:
*"si muevo un poquito este peso, ¿el error sube o baja, y cuánto?"*. Esa respuesta es el **gradiente**.
La regla de la cadena permite calcularlo capa por capa **desde la salida hacia la entrada**
(de ahí "hacia atrás"). Después movemos cada peso un pasito en dirección contraria al gradiente:
es el **descenso de gradiente**.

- **Qué hace:**
    - `entropia_cruzada(y_real, prob)` mide cuánto «duele» cada predicción: castiga mucho que la red dé una probabilidad baja al grupo correcto. `np.clip` evita calcular `log(0)`.
    - Los tres `print` la prueban con casos conocidos: siempre 0,5; acertar con seguridad y fallar con seguridad.
- **Qué debe salir:**
    - `Si predices siempre 0.5, la pérdida es 0.693 (= ln 2)`, `Si aciertas con seguridad (0.99 / 0.01): 0.01` y `Si fallas con seguridad (0.01 / 0.99): 4.605`.
- **Cómo leerlo:**
    - Menos pérdida = mejor. Fallar con mucha seguridad se castiga muchísimo más (4,605) que acertar con seguridad (0,01).
- **Si falla:**
    - `NameError: name 'np' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué pérdida tiene una red que siempre dice 0,5?* → **0,693** (= ln 2): es el punto de partida de una red sin entrenar.
2. *¿Por qué fallar con seguridad cuesta 4,605 y acertar con seguridad solo 0,01?* → Porque la pérdida usa el **logaritmo** de la probabilidad que se dio al grupo correcto: si es muy baja (0,01), el logaritmo es muy grande.
3. *¿Qué quiere decir que la pérdida baje durante el entrenamiento?* → Que las predicciones de la red se acercan cada vez más a las respuestas reales.

</details>

#### A8 · Ejercicio 3 — Un paso de descenso de gradiente

- **Qué hace:**
    - `{k: parametros[k] - tasa * gradientes[k] for k in parametros}` recorre cada parámetro (`W1`, `b1`, `W2`, `b2`) y le resta `tasa` veces su gradiente: da un pasito **en contra** de la dirección en la que sube el error.
    - Como se construye un diccionario nuevo, el original no se modifica.
    - La comprobación usa `W1 = [1, 2]` con gradiente `[0,5, −1]` y tasa 0,1: debe salir `[0,95, 2,1]`; y `b1 = 0,5` con gradiente 1 debe salir 0,4.
- **Qué debe salir:**
    - `✔ Ejercicio 3 correcto`.
- **Si falla:**
    - `Cada parámetro debe restar tasa * gradiente` → has sumado en vez de restar, o has olvidado multiplicar por `tasa`.
    - `No modifiques el diccionario original: devuelve uno nuevo` → has cambiado `parametros[k]` en su sitio (`+=`/`-=` o asignación) en vez de construir un diccionario nuevo.
    - `NameError: name 'np' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué valores salen para `W1` y `b1` en la prueba?* → `W1 = [0,95, 2,1]` y `b1 = [0,4]`.
2. *¿Por qué se **resta** el gradiente y no se suma?* → Porque el gradiente apunta hacia donde el error **sube**; para bajarlo hay que ir en sentido contrario.
3. *¿Qué pasaría con una tasa de aprendizaje enorme?* → Que los pasos serían tan grandes que se **saltaría** el mínimo y el error subiría en vez de bajar (se verá en el mini-reto).

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
def paso_gradiente(parametros, gradientes, tasa):
    return {k: parametros[k] - tasa * gradientes[k] for k in parametros}
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>